In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ====================== 1. Load Training Data ======================
# Assume the training set has been preprocessed and saved as a CSV file
# The training features CSV should include all feature columns (without the label)
train_csv = 'X_train_processed.csv'  # Training features CSV
X_train_df = pd.read_csv(train_csv)

# Extract feature matrix
X_train = X_train_df.values
n_train, p = X_train.shape

# ====================== 2. Standardize Training Data ======================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# ====================== 3. Load New Samples ======================
new_csv = 'X_new.csv'  # New samples CSV, same columns as training set
X_new_df = pd.read_csv(new_csv)
X_new = X_new_df.values

# ====================== 4. Standardize New Samples ======================
X_new_scaled = scaler.transform(X_new)

# ====================== 5. Compute Leverage ======================
def compute_leverage(X_train_scaled, X_new_scaled):
    """
    X_train_scaled: n_train x p, standardized training features
    X_new_scaled: n_new x p, standardized new samples
    Returns leverage values for each new sample
    """
    n_train = X_train_scaled.shape[0]
    p = X_train_scaled.shape[1]
    # Add intercept column
    X_aug = np.hstack([np.ones((n_train, 1)), X_train_scaled])
    XtX_inv = np.linalg.inv(X_aug.T @ X_aug)
    
    # Add intercept column to new samples
    X_new_aug = np.hstack([np.ones((X_new_scaled.shape[0], 1)), X_new_scaled])
    
    # Compute leverage for each new sample
    h_new = np.sum(X_new_aug @ XtX_inv * X_new_aug, axis=1)
    return h_new

leverage_new = compute_leverage(X_train_scaled, X_new_scaled)

# ====================== 6. Compute AD Threshold ======================
h_threshold = 3 * (p + 1) / n_train
print(f"Leverage threshold h*: {h_threshold:.6f}")

# ====================== 7. Determine if New Samples are in AD ======================
in_AD = leverage_new <= h_threshold
X_new_df['Leverage'] = leverage_new
X_new_df['In_AD'] = in_AD

# Output results
print("Proportion of new samples within AD:", np.mean(in_AD))
print(X_new_df.head())

# ====================== 8. Save Results ======================
X_new_df.to_csv('X_new_with_AD.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

# ====================== 1. Load Training Data ======================
# Features CSV for training set (without labels)
train_csv = 'X_train_processed.csv'
X_train_df = pd.read_csv(train_csv)
X_train = X_train_df.values
n_train, p = X_train.shape

# ====================== 2. Standardize Training Data ======================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# ====================== 3. Load New Sample Features ======================
new_csv = 'X_new.csv'
X_new_df = pd.read_csv(new_csv)
X_new = X_new_df.values
X_new_scaled = scaler.transform(X_new)

# ====================== 4. Compute Leverage for New Samples ======================
def compute_leverage(X_train_scaled, X_new_scaled):
    n_train, p = X_train_scaled.shape
    X_aug = np.hstack([np.ones((n_train, 1)), X_train_scaled])
    XtX_inv = np.linalg.inv(X_aug.T @ X_aug)
    X_new_aug = np.hstack([np.ones((X_new_scaled.shape[0], 1)), X_new_scaled])
    h_new = np.sum(X_new_aug @ XtX_inv * X_new_aug, axis=1)
    return h_new

leverage_new = compute_leverage(X_train_scaled, X_new_scaled)
h_threshold = 3 * (p + 1) / n_train
in_AD_leverage = leverage_new <= h_threshold

# ====================== 5. Load Training and New Sample SMILES ======================
train_smiles_csv = 'train_smiles.csv'
new_smiles_csv = 'new_smiles.csv'
train_smiles = pd.read_csv(train_smiles_csv)['SMILES'].tolist()
new_smiles = pd.read_csv(new_smiles_csv)['SMILES'].tolist()

# Convert SMILES to RDKit molecules
train_mols = [Chem.MolFromSmiles(s) for s in train_smiles]
new_mols = [Chem.MolFromSmiles(s) for s in new_smiles]

# ====================== 6. Compute Murcko Scaffold Fingerprints ======================
# Compute Morgan fingerprints for training molecules
train_fps = [AllChem.GetMorganFingerprintAsBitVect(m, radius=2, nBits=1024) for m in train_mols]

# Compute fingerprints for new molecules
new_fps = [AllChem.GetMorganFingerprintAsBitVect(m, radius=2, nBits=1024) for m in new_mols]

# ====================== 7. Compute Maximum Tanimoto Similarity ======================
max_sim = []
for fp_new in new_fps:
    sims = DataStructs.BulkTanimotoSimilarity(fp_new, train_fps)
    max_sim.append(max(sims))

# Set similarity threshold (e.g., 0.6)
similarity_threshold = 0.6
in_AD_structure = np.array(max_sim) >= similarity_threshold

# ====================== 8. Combine Dual AD ======================
# True if sample is in both feature-space AD and chemical-space AD
in_AD_dual = in_AD_leverage & in_AD_structure

# ====================== 9. Save Results ======================
X_new_df['Leverage'] = leverage_new
X_new_df['In_AD_Feature'] = in_AD_leverage
X_new_df['Max_Tanimoto'] = max_sim
X_new_df['In_AD_Structure'] = in_AD_structure
X_new_df['In_AD_Dual'] = in_AD_dual

print("Proportion of new samples in feature-space AD:", np.mean(in_AD_leverage))
print("Proportion of new samples in structure-space AD:", np.mean(in_AD_structure))
print("Proportion of new samples in dual AD:", np.mean(in_AD_dual))

X_new_df.to_csv('X_new_with_dual_AD.csv', index=False)